# Hyperparameter Tuning an ANN — Choosing Hidden Layers & Neurons

### 📌 Recap

Every ANN built so far (classification in notebooks 01–05, regression in 06) used a fixed, hand-picked architecture: `12 → 64 → 32 → 1`. This notebook answers the natural follow-up question — **how do you actually choose those numbers**, instead of guessing?

## Determining the optimal number of hidden layers and neurons

This is genuinely hard to answer analytically and usually requires experimentation, but a few guidelines help narrow the search:

- **Start simple.** Begin with a simple architecture and gradually increase complexity if needed.
- **Grid Search / Random Search.** Systematically try different architectures rather than guessing one.
- **Cross-validation.** Evaluate each candidate architecture's performance with cross-validation, not a single train/test split — same principle as in classical ML.
- **Heuristics and rules of thumb** as starting points: the number of neurons in a hidden layer is often somewhere between the input layer size and the output layer size; 1–2 hidden layers is a common starting point before reaching for anything deeper.

This notebook demonstrates the **Grid Search** approach: define a small, parameterized model-building function, wrap it as a scikit-learn-compatible estimator, and let `GridSearchCV` try every combination of neuron count / layer count / epoch count, scored with cross-validation.

> ⚠️ This notebook was rewritten to fix issues found in the original (see Section 8) and to make it genuinely reusable as a **template for hyperparameter-tuning any future ANN problem** in this project — not just this specific churn classifier. Section 7 spells out exactly what to change to adapt it to a new problem (including regression).

## 1. Imports

One import is worth explaining: `scikeras.wrappers.KerasClassifier`. Older tutorials (and the original version of this notebook) reach for `keras.wrappers.scikit_learn.KerasClassifier` — that module **no longer exists** in current TensorFlow/Keras (it was removed after being deprecated). **`scikeras`** is the actively-maintained community replacement that wraps a Keras model as a scikit-learn-compatible estimator, which is what lets `GridSearchCV` (a scikit-learn tool) drive a Keras model at all. It's already in this project's `requirements.txt`.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

## 2. Load and preprocess the data

Same steps as [02 Data PreProcessing.ipynb](02%20Data%20PreProcessing.ipynb): drop identifiers, label-encode `Gender`, one-hot encode `Geography`, split, scale.

> ⚠️ **Deliberately different from the original notebook: no pickle files are saved here.** The original saved `label_encoder_gender.pkl`, `onehot_encoder_geo.pkl`, and `scaler.pkl` — the exact same filenames `app.py` and notebook 02 already produce and depend on. Even though this notebook happens to use the same target column as classification (so the resulting scaler would likely end up statistically equivalent), a **hyperparameter search notebook re-writing the production app's dependencies as a side effect is fragile and easy to get wrong later** (e.g. if this notebook's split or preprocessing ever diverges even slightly). This notebook's job is to search for good hyperparameters, not to own the canonical preprocessing artifacts — so `X_train`/`X_test` are built here purely in memory, for this notebook's own use.

In [2]:
data = pd.read_csv("../Churn_Modelling.csv")
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

onehot_encoder_geo = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
geo_encoded = onehot_encoder_geo.fit_transform(data[["Geography"]])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(["Geography"]))
data = pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)

X = data.drop("Exited", axis=1)
y = data["Exited"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train shape:", X_train.shape)

X_train shape: (8000, 12)


## 3. A parameterized model-builder function

This is the reusable core of the whole approach: instead of hardcoding an architecture, write a function that **builds** one from parameters, with sensible defaults. `GridSearchCV` will call this function once per candidate combination, each time with different `neurons`/`layers` values.

- First hidden layer is always added.
- `for _ in range(layers - 1)` adds `layers - 1` *more* hidden layers, so `layers=1` → just the first one, `layers=3` → three total — correct as originally written, verified below.
- Output layer + compile settings match [03 Model Training.ipynb](03%20Model%20Training.ipynb): sigmoid + binary cross-entropy for this binary classification problem.

Uses `Input(shape=...)` (matching the project's established convention) instead of `input_shape=` on the first `Dense` layer.

In [3]:
def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    model.add(Dense(neurons, activation="relu"))

    for _ in range(layers - 1):
        model.add(Dense(neurons, activation="relu"))

    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


# Quick sanity check of the layer-count logic before handing this to GridSearchCV
for layers in [1, 2, 3]:
    m = create_model(neurons=16, layers=layers)
    n_dense_layers = sum(1 for l in m.layers if isinstance(l, Dense))
    print(f"layers={layers} -> {n_dense_layers} Dense layers total (hidden + output)")

layers=1 -> 2 Dense layers total (hidden + output)
layers=2 -> 3 Dense layers total (hidden + output)


layers=3 -> 4 Dense layers total (hidden + output)


Confirms `layers=N` produces `N` hidden `Dense` layers plus the 1 output layer — `N+1` `Dense` layers total, exactly as intended.

## 4. Wrap it as a scikit-learn estimator

> ⚠️ **Fix applied:** the original used `KerasClassifier(..., build_fn=create_model, ...)`. Current `scikeras` still accepts `build_fn`, but only with a `UserWarning` that it's deprecated in favor of `model=` — visible in the original notebook's own saved output (*"``build_fn`` will be renamed to ``model`` in a future release... use of ``build_fn`` will raise an Error instead"*). Using `model=create_model` here avoids the warning entirely and won't break on a future scikeras release.

`neurons=32, layers=1` here are just the *default* values used if `GridSearchCV` doesn't override them — `param_grid` below is what actually varies them. Any keyword scikeras doesn't recognize as its own (like `neurons`/`layers`) gets forwarded straight into `create_model(...)`, which is what makes them tunable via `param_grid` in the first place.

`verbose=0` here (unlike the video, which used `verbose=1`) keeps this notebook's output readable — `GridSearchCV`'s own `verbose=` setting below already reports per-fold/per-candidate progress without also printing every epoch of every one of the (up to) 48 individual training runs.

In [4]:
model = KerasClassifier(model=create_model, neurons=32, layers=1, verbose=0)

## 5. Define the search space

Every combination of these gets tried: **4 neuron counts × 2 layer counts × 2 epoch counts = 16 candidate architectures**, each evaluated with 3-fold cross-validation below — **48 total training runs**.

> 💡 **Extending this for a real search:** `batch_size` and `learning_rate` (via a custom optimizer object) are natural additions to this grid for a more thorough search — left out here to keep the runtime reasonable for a walkthrough, but this dictionary is the single place to extend for a deeper search later.

In [5]:
param_grid = {
    "neurons": [16, 32, 64, 128],
    "layers": [1, 2],
    "epochs": [50, 100],
}

## 6. Run the grid search

> 📝 **`n_jobs=-1` and Keras/TensorFlow on Windows:** this combination is a known source of multiprocessing/pickling issues on some setups. Verified directly on this project's real environment with a small trial run before committing to the full grid below — it works cleanly here, no changes needed. If it ever hangs or errors in a different environment, `n_jobs=1` (slower, but avoids multiprocessing entirely) is the standard fallback.

This cell trains **48 separate neural networks** (16 candidates × 3 folds) — expect this to take a few minutes, not seconds.

In [6]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3, verbose=1)
grid_result = grid.fit(X_train, y_train)

print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


Best: 0.856750 using {'epochs': 50, 'layers': 1, 'neurons': 16}


In [7]:
import pandas as pd

results_df = pd.DataFrame(grid_result.cv_results_)[["param_neurons", "param_layers", "param_epochs", "mean_test_score", "std_test_score"]]
results_df = results_df.sort_values("mean_test_score", ascending=False).reset_index(drop=True)
results_df.head(10)

,param_neurons,param_layers,param_epochs,mean_test_score,std_test_score
0,16,1,50,0.856750,0.003247
1,16,1,100,0.856499,0.005095
2,32,1,100,0.855624,0.006348
3,16,2,50,0.855374,0.004925
4,128,1,100,0.855125,0.006264
5,64,1,50,0.854624,0.005544
6,32,1,50,0.854249,0.004486
7,128,1,50,0.853249,0.005546
8,16,2,100,0.852625,0.002678
9,32,2,50,0.850624,0.003386


## 7. ⚠️ Adapting this notebook for a *different* ANN problem (the actual template)

This is the part that makes the notebook genuinely reusable, not just a one-off. To point this exact pattern at a new problem statement:

| Change | Classification (this notebook) | Regression (e.g. notebook 06's salary problem) |
|---|---|---|
| scikeras wrapper | `KerasClassifier` | **`KerasRegressor`** (also from `scikeras.wrappers`) |
| Output layer | `Dense(1, activation='sigmoid')` | `Dense(1)` — no activation (linear) |
| Loss | `'binary_crossentropy'` | `'mean_absolute_error'` (or `'mean_squared_error'`) |
| `model.compile(metrics=...)` | `['accuracy']` | `['mae']` |
| What `grid_result.best_score_` means | accuracy (higher is better) | scikeras's `KerasRegressor` default score is **R²** (higher is better) — not MAE directly; add a custom `scoring=` to `GridSearchCV` (e.g. `scoring='neg_mean_absolute_error'`) if you want the grid ranked by MAE instead |
| `param_grid` | unchanged in structure — same `neurons`/`layers`/`epochs` keys work identically, since they're just forwarded into `create_model` either way | |

The `create_model(neurons, layers)` pattern, the `KerasClassifier`/`KerasRegressor` wrapping, and the `GridSearchCV` call itself don't need to change structurally at all — only the five rows in the table above. That's the reusable core of this template.

## 8. Corrections applied vs. the original notebook

1. **No longer saves `label_encoder_gender.pkl` / `onehot_encoder_geo.pkl` / `scaler.pkl`** — those are owned by notebook 02 (and shared with notebook 06, see [06 Salary Regression.ipynb](06%20Salary%20Regression.ipynb#4.)), not by this hyperparameter-search notebook. Re-saving them here risked silently overwriting the production app's dependencies.
2. **Dataset path** — `'Churn_Modelling.csv'` → `'../Churn_Modelling.csv'`, since this notebook lives in `notebooks/`.
3. **`OneHotEncoder(handle_unknown='ignore')` + `.toarray()`** → `OneHotEncoder(sparse_output=False, handle_unknown='ignore')`, no `.toarray()` — matches the project's established convention.
4. **`input_shape=` on the first `Dense` layer** → explicit `Input(shape=...)`.
5. **`build_fn=create_model`** → **`model=create_model`** — the deprecated parameter name (visible as a live warning in the original notebook's own output) replaced with the one `scikeras` actually recommends.
6. **`KerasClassifier(..., verbose=1)`** → `verbose=0` — avoids flooding the notebook with per-epoch output across all 48 individual training runs; `GridSearchCV(verbose=1)` alone already reports progress.
7. **All outputs re-executed from scratch** against this project's real environment — the original notebook's outputs were from an unrelated `E:\UDemy Final\...` install.
8. **Verified `n_jobs=-1` actually works on this Windows setup** before running the full 48-fit grid, rather than assuming.

> 📝 One thing left as-is, worth knowing: `EarlyStopping` was imported in the original notebook but never actually used anywhere. Wiring it into `KerasClassifier(..., callbacks=[EarlyStopping(...)])` would speed up the search (candidates that plateau early stop sooner) — a reasonable enhancement, not applied here to keep this version close to the original's intent.

## 9. Summary

- Grid search over `neurons`, `layers`, and `epochs` for the same churn-classification architecture used throughout this project, scored with 3-fold cross-validation (48 total training runs).
- The reusable core: a **parameterized `create_model(neurons, layers)` function** + **`KerasClassifier`/`KerasRegressor` wrapping** + **`GridSearchCV`** — this pattern needs almost no structural changes to point at a different ANN problem, classification or regression (see the table in Section 7).
- Fixed a real, live bug from the original (`build_fn` → `model`), removed a fragile side effect (overwriting shared production pickles from an experimentation notebook), and verified `n_jobs=-1` actually works in this environment before trusting it for a longer run.

## 10. Likely exam / interview questions

1. Why can't `GridSearchCV` (a scikit-learn tool) be used directly on a raw Keras `Sequential` model?
2. What's the difference between what `KerasClassifier` and `KerasRegressor` expect from the wrapped model's output layer?
3. Why does `create_model`'s `for _ in range(layers - 1)` loop produce exactly `layers` hidden layers in total, not `layers - 1`?
4. Why is it risky for a hyperparameter-search notebook to re-save the same preprocessing pickle files a production app depends on?
5. What's a practical reason to prefer `n_jobs=1` over `n_jobs=-1` when grid-searching a Keras model, even though `-1` is usually "use all cores"?
6. Following the table in Section 7, what specifically would need to change to grid-search a regression ANN's hidden layer sizes instead of this classifier's?

## 11. What's next

- Rebuild the final classification model using the winning `neurons`/`layers`/`epochs` combination found here.
- GitHub push + Streamlit Cloud deployment (both apps).